# 3D toric code — L=2 **OBC** exact-diagonalization ground truth (self-contained)

Matrix-free Krylov diagonalization of the perturbed **3D toric code** with **open
boundary conditions (OBC)** at $L=2$, swept over a 5×5 grid of fields
$h_x,h_z\in\{0,0.1,0.2,0.3,0.4\}$. Produces per-point reference JSONs (same schema
as `Three_TC/validation.py` expects) so OBC NQS runs can be scored against exact
$E_0$, gap, $\langle A_v\rangle$, $\langle B_p\rangle$, $\langle\sigma_x\rangle$,
$\langle\sigma_z\rangle$.

$$H = -J\sum_v A_v - J\sum_p B_p - h_x\sum_i \sigma^x_i - h_z\sum_i \sigma^z_i$$

**OBC convention (matches `Three_TC/model/geometry.py`).** Qubits sit on the edges
*inside* the open box $[0,L-1]^3$, so $N = 3L^3 - 3L^2 = 12$ at $L=2$
($2^{12}=4096$ states — trivial). Vertex stars $A_v$ at the boundary are
**truncated** (fewer than 6 edges; the missing legs are simply absent). Plaquettes
$B_p$ are kept **only when the full 4-edge face exists** — an incomplete boundary
face is not a stabilizer and is dropped. This is exactly the model the OBC NQS
pipeline trains, so ED and NQS use identical `vertex_all` / `plaq_all`.

Self-contained: pure numpy + scipy, no repo imports. Drop into Colab and *Run all*.

In [ ]:
# Colab already ships numpy/scipy/matplotlib; this is a no-op if present.
!pip install -q numpy scipy matplotlib

In [ ]:
import json, time, os
from collections import defaultdict
import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

## Geometry — netket-free 3D toric code (OBC)

Edges live at half-integer coordinates $v+\tfrac12\hat e_c$. We keep only those
inside the closed box $[0,L-1]^3$ (OBC), key them by their $2\times$-integer
coordinate (avoids float-equality pitfalls), and build vertex stars / complete
plaquettes by coordinate lookup (`-1` ⇒ edge outside the box).

In [ ]:
class ThreeD_ToricCodeGeometry_OBC:
    """Open-boundary 3D toric code on an Lx×Ly×Lz cubic lattice (qubits on edges).

    Faithful port of the OBC branch of Three_TC/model/geometry.py:
      - N = 3 Lx Ly Lz - (Lx Ly + Lx Lz + Ly Lz),
      - vertex_all: 6-neighbour stars, truncated at the boundary (-1 padding),
      - plaq_all:   only complete 4-edge faces (incomplete boundary faces dropped).
    """
    def __init__(self, Lx, Ly, Lz):
        self.Lx, self.Ly, self.Lz = Lx, Ly, Lz
        self.N = 3*Lx*Ly*Lz - (Lx*Ly + Lx*Lz + Ly*Lz)
        self._build_coords()
        self.vertex_all = self._stabilizers_vertex()
        self.plaq_all = self._stabilizers_plaquette()

    def _build_coords(self):
        e = np.eye(3)
        coords = []
        for c in range(3):                       # edge orientation
            for ix in range(self.Lx):
                for iy in range(self.Ly):
                    for iz in range(self.Lz):
                        m = np.array([ix, iy, iz], float) + 0.5*e[c]
                        if (m <= np.array([self.Lx-1, self.Ly-1, self.Lz-1])).all():
                            coords.append(tuple(m))
        coords.sort(key=lambda c: (c[2], c[1], c[0]))   # lexsort(z,y,x), repo order
        self.arr_coord = np.array(coords)
        assert len(coords) == self.N, (len(coords), self.N)
        self._coord_to_idx = {tuple((2*np.asarray(c)).round().astype(int)): i
                              for i, c in enumerate(self.arr_coord)}

    def _idx(self, coord):
        """Edge index at midpoint `coord`; -1 if outside the open box."""
        key = tuple((2*np.asarray(coord)).round().astype(int))
        return self._coord_to_idx.get(key, -1)

    def _stabilizers_vertex(self):
        e = np.eye(3); out = []
        for ix in range(self.Lx):
            for iy in range(self.Ly):
                for iz in range(self.Lz):
                    v = np.array([ix, iy, iz], float)
                    nbrs = [self._idx(v + 0.5*s*e[ax])
                            for ax in range(3) for s in (+1, -1)]
                    out.append(nbrs)          # truncated stars keep -1 padding
        return out

    def _stabilizers_plaquette(self):
        e = np.eye(3); out = []
        for c in range(3):                    # plaquette normal axis
            a, b = [i for i in range(3) if i != c]
            edge_offs = [+0.5*e[a], -0.5*e[a], +0.5*e[b], -0.5*e[b]]
            for ix in range(self.Lx):
                for iy in range(self.Ly):
                    for iz in range(self.Lz):
                        ctr = np.array([ix, iy, iz], float) + 0.5*e[a] + 0.5*e[b]
                        edges = [self._idx(ctr + o) for o in edge_offs]
                        if -1 in edges:
                            continue          # incomplete face -> not a stabilizer
                        out.append(edges)
        return out

## Matrix-free Hamiltonian and Pauli-string expectations

$H\psi$ is computed as a function (scipy `LinearOperator`), so peak memory is a few
length-$2^N$ vectors. `mask()` skips `-1`, so truncated vertex stars are handled
automatically.

In [ ]:
def mask(qubits):
    """Bitmask over qubit indices; skips -1 (OBC boundary padding)."""
    m = 0
    for q in qubits:
        if q != -1:
            m |= 1 << int(q)
    return m

def z_string_eigvals(basis, m, N):
    """(-1)^popcount(b & m) per basis state b, as float64."""
    parity = np.zeros_like(basis, dtype=np.int8)
    for i in range(N):
        if m & (1 << i):
            parity ^= ((basis >> i) & 1).astype(np.int8)
    return 1.0 - 2.0 * parity.astype(np.float64)

def make_hamiltonian_op(geo, hx=0.0, hz=0.0, J=1.0):
    N = geo.N; dim = 1 << N
    basis = np.arange(dim, dtype=np.int64)
    diag = np.zeros(dim, dtype=np.float64)
    for p in geo.plaq_all:                       # -J sum_p B_p (Z plaquettes)
        diag -= J * z_string_eigvals(basis, mask(p), N)
    if hz != 0:
        for i in range(N):
            diag -= hz * z_string_eigvals(basis, 1 << i, N)
    x_strings = defaultdict(float)               # off-diagonal X-strings
    for v in geo.vertex_all:                     # -J sum_v A_v (X stars)
        x_strings[mask(v)] -= J
    if hx != 0:
        for i in range(N):
            x_strings[1 << i] -= hx
    x_strings = {m: c for m, c in x_strings.items() if c != 0 and m != 0}
    xm = np.fromiter(x_strings.keys(), dtype=np.int64, count=len(x_strings))
    xc = np.fromiter(x_strings.values(), dtype=np.float64, count=len(x_strings))

    def matvec(psi):
        psi = np.asarray(psi, dtype=np.float64)
        out = diag * psi
        for k in range(xm.shape[0]):
            out = out + xc[k] * psi[basis ^ xm[k]]
        return out
    return spla.LinearOperator((dim, dim), matvec=matvec, dtype=np.float64), basis

def expect_x_string(psi, basis, m):
    return float(np.real(np.sum(psi * psi[basis ^ m]))) if m else 1.0

def expect_z_string(psi, basis, m, N):
    return float(np.sum(psi**2 * z_string_eigvals(basis, m, N))) if m else 1.0

## Self-checks

Confirm the OBC counts, that every $A_v$ commutes with every $B_p$ (even support
overlap), and the unperturbed ($h_x=h_z=0$) energy equals $-(N_v^{\text{full}} +
N_p)$ … here we just print the pieces and the zero-field $E_0$.

In [ ]:
geo = ThreeD_ToricCodeGeometry_OBC(2, 2, 2)
print("N qubits   =", geo.N)
print("N vertices =", len(geo.vertex_all), "(stars, truncated at boundary)")
print("N plaquettes =", len(geo.plaq_all), "(complete faces only)")
assert geo.N == 12 and len(geo.plaq_all) == 6 and len(geo.vertex_all) == 8
assert all(-1 not in p for p in geo.plaq_all), "plaq_all must have no -1"

# A_v / B_p commute  <=>  |support overlap| even
psets = [set(p) for p in geo.plaq_all]
ok = all(len(set(x for x in v if x != -1) & ps) % 2 == 0
         for v in geo.vertex_all for ps in psets)
print("all A_v, B_p commute:", ok); assert ok

H0, basis0 = make_hamiltonian_op(geo, hx=0.0, hz=0.0, J=1.0)
e0 = spla.eigsh(H0, k=1, which="SA", return_eigenvectors=False)[0]
print("zero-field E0 =", round(float(e0), 6))

## 5×5 sweep over $(h_x, h_z)$

For each grid point: Lanczos for the two lowest levels, ground-state observables,
and a per-point JSON `exact_diag_L2_OBC_hx{hx}_hz{hz}.json` matching the validation
reference schema.

In [ ]:
HX_LIST = [0.0, 0.1, 0.2, 0.3, 0.4]
HZ_LIST = [0.0, 0.1, 0.2, 0.3, 0.4]
J = 1.0
OUT_DIR = "outputs_obc_ed"
os.makedirs(OUT_DIR, exist_ok=True)

geo = ThreeD_ToricCodeGeometry_OBC(2, 2, 2)
N, Nv, Np = geo.N, len(geo.vertex_all), len(geo.plaq_all)
v_masks = [mask(v) for v in geo.vertex_all]
p_masks = [mask(p) for p in geo.plaq_all]

grid = {key: np.zeros((len(HX_LIST), len(HZ_LIST)))
        for key in ("E0", "E1", "gap", "A_v_mean", "B_p_mean", "sx_mean", "sz_mean")}

t_all = time.time()
for ix, hx in enumerate(HX_LIST):
    for iz, hz in enumerate(HZ_LIST):
        H, basis = make_hamiltonian_op(geo, hx=hx, hz=hz, J=J)
        evals, evecs = spla.eigsh(H, k=2, which="SA")
        order = np.argsort(evals); evals = evals[order]
        psi0 = evecs[:, order[0]]; psi0 /= np.linalg.norm(psi0)
        E0, E1 = float(evals[0]), float(evals[1]); gap = E1 - E0

        A_v = float(np.mean([expect_x_string(psi0, basis, m) for m in v_masks]))
        B_p = float(np.mean([expect_z_string(psi0, basis, m, N) for m in p_masks]))
        sx = float(np.mean([expect_x_string(psi0, basis, 1 << i) for i in range(N)]))
        sz = float(np.mean([expect_z_string(psi0, basis, 1 << i, N) for i in range(N)]))

        for k, val in (("E0", E0), ("E1", E1), ("gap", gap), ("A_v_mean", A_v),
                       ("B_p_mean", B_p), ("sx_mean", sx), ("sz_mean", sz)):
            grid[k][ix, iz] = val

        result = {"model": "bosonic", "bc": "OBC",
                  "Lx": 2, "Ly": 2, "Lz": 2, "N": N,
                  "N_vertices": Nv, "N_plaquettes": Np,
                  "hx": hx, "hz": hz, "hy": 0.0, "J": J, "dtype": "float64",
                  "E0": E0, "E1": E1, "gap": gap,
                  "A_v_mean": A_v, "B_p_mean": B_p, "sx_mean": sx, "sz_mean": sz}
        fname = os.path.join(OUT_DIR, f"exact_diag_L2_OBC_hx{hx}_hz{hz}.json")
        with open(fname, "w") as f:
            json.dump(result, f, indent=2)
        print(f"hx={hx:.1f} hz={hz:.1f}  E0={E0:+.5f}  gap={gap:.4f}  "
              f"<A_v>={A_v:.3f} <B_p>={B_p:.3f}", flush=True)

print(f"\nDone in {time.time()-t_all:.1f}s -> {len(HX_LIST)*len(HZ_LIST)} JSONs in {OUT_DIR}/")

## Plots — $E_0$ and gap heatmaps over the $(h_x,h_z)$ grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ext = [HZ_LIST[0], HZ_LIST[-1], HX_LIST[0], HX_LIST[-1]]
for ax, key, title in ((axes[0], "E0", r"$E_0$"), (axes[1], "gap", r"gap $E_1-E_0$")):
    im = ax.imshow(grid[key], origin="lower", extent=ext, aspect="auto", cmap="viridis")
    ax.set_xlabel(r"$h_z$"); ax.set_ylabel(r"$h_x$"); ax.set_title(title + "  (L=2 OBC)")
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()